# 试题2：请用OpenCV中的python接口实现基于大津阈值法的图像二值化分割。（初级）

大津阈值法（Otsu's Method），又称最大类间方差法，是一种自动确定图像二值化阈值的方法。
其核心思想是使前景和背景的类间方差最大化。

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageSequence
import os

## 定义辅助函数

In [ ]:
def load_multipage_tiff(path):
    """加载多页TIFF图像"""
    return np.array([np.array(p) for p in ImageSequence.Iterator(Image.open(path))])

def otsu_thresholding(image):
    """
    使用大津阈值法进行图像二值化分割
    
    参数:
        image: 输入图像（灰度图）
    
    返回:
        binary_image: 二值化后的图像
        threshold: 自动计算的最佳阈值
    """
    # 使用OpenCV的大津阈值法
    # thresh=0表示自动计算阈值
    # maxval=255表示二值化后的最大值为255
    # type=cv2.THRESH_OTSU表示使用大津阈值法
    ret, binary = cv2.threshold(src=image, thresh=0, maxval=255, type=cv2.THRESH_OTSU)
    
    return binary, ret

def manual_otsu(image):
    """
    手动实现大津阈值法（用于理解算法原理）
    
    参数:
        image: 输入图像（灰度图）
    
    返回:
        binary_image: 二值化后的图像
        threshold: 自动计算的最佳阈值
    """
    # 计算直方图
    histogram = cv2.calcHist([image], [0], None, [256], [0, 256])
    histogram = histogram.flatten()
    
    # 归一化直方图
    histogram = histogram / float(image.size)
    
    # 初始化变量
    best_threshold = 0
    max_variance = 0
    
    # 遍历所有可能的阈值
    for threshold in range(256):
        # 计算背景和前景的概率
        w0 = np.sum(histogram[:threshold])
        w1 = np.sum(histogram[threshold:])
        
        if w0 == 0 or w1 == 0:
            continue
        
        # 计算背景和前景的均值
        u0 = np.sum([i * histogram[i] for i in range(threshold)]) / w0
        u1 = np.sum([i * histogram[i] for i in range(threshold, 256)]) / w1
        
        # 计算类间方差
        variance = w0 * w1 * (u0 - u1) ** 2
        
        # 更新最佳阈值
        if variance > max_variance:
            max_variance = variance
            best_threshold = threshold
    
    # 应用最佳阈值进行二值化
    _, binary = cv2.threshold(image, best_threshold, 255, cv2.THRESH_BINARY)
    
    return binary, best_threshold

def visualize_otsu_result(original_image, binary_image, threshold):
    """
    可视化大津阈值法的结果
    
    参数:
        original_image: 原始图像
        binary_image: 二值化后的图像
        threshold: 使用的阈值
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 显示原图
    axes[0].imshow(original_image, cmap='gray')
    axes[0].set_title('Source Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # 显示二值化图像
    axes[1].imshow(binary_image, cmap='gray')
    axes[1].set_title(f'OTSU Binary Image\n(Threshold = {threshold:.2f})', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # 显示直方图和阈值位置
    axes[2].hist(original_image.ravel(), bins=256, range=[0, 256], color='steelblue', alpha=0.7, edgecolor='black')
    axes[2].axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.2f}')
    axes[2].set_title('Histogram with Threshold', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Pixel Value', fontsize=12)
    axes[2].set_ylabel('Frequency', fontsize=12)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

## 主程序

In [ ]:
# 数据集路径
data_dir = './data'
train_volume_path = os.path.join(data_dir, 'train-volume.tif')

# 加载训练集图像
print("正在加载训练集图像...")
images = load_multipage_tiff(train_volume_path)
print(f"训练集图像形状: {images.shape}")

# 选择第一张图像进行分析
image = images[0]
print(f"第一张图像形状: {image.shape}")

## 使用OpenCV内置的大津阈值法

In [ ]:
print("\n使用OpenCV内置的大津阈值法...")
binary_cv2, threshold_cv2 = otsu_thresholding(image)
print(f"自动计算的最佳阈值: {threshold_cv2}")

## 手动实现大津阈值法（用于验证）

In [ ]:
print("\n手动实现大津阈值法...")
binary_manual, threshold_manual = manual_otsu(image)
print(f"手动计算的最佳阈值: {threshold_manual}")

# 比较两种方法的结果
print(f"\n阈值比较:")
print(f"  OpenCV阈值: {threshold_cv2}")
print(f"  手动阈值: {threshold_manual}")
print(f"  差异: {abs(threshold_cv2 - threshold_manual)}")

## 可视化结果

In [ ]:
print("\n生成可视化结果...")
fig = visualize_otsu_result(image, binary_cv2, threshold_cv2)
fig.savefig('./思考题/otsu_result.png', dpi=300, bbox_inches='tight')
print("结果已保存到: ./思考题/otsu_result.png")
plt.show()

## 分割结果分析

In [ ]:
print("\n分割结果分析:")
print("  - 大津阈值法自动找到了最佳阈值")
print("  - 通过观察分割结果，我们可以看到:")
print("    * 一些噪声被错误地分割为前景")
print("    * 某些目标区域没有被完整分割")
print("    * 边界处的分割效果不理想")
print("  - 这说明对于该数据集，传统的阈值分割方法效果有限")
print("  - 需要使用更复杂的深度学习方法（如U-Net）来获得更好的分割效果")

# 计算分割质量指标
foreground_pixels = np.sum(binary_cv2 == 255)
background_pixels = np.sum(binary_cv2 == 0)
total_pixels = image.size

print(f"\n分割统计:")
print(f"  前景像素数: {foreground_pixels} ({foreground_pixels/total_pixels*100:.2f}%)")
print(f"  背景像素数: {background_pixels} ({background_pixels/total_pixels*100:.2f}%)")